# Micrograd from scratch

Learning project on backpropagation by implementing it from scratch. This is a follow-along implementation of Andrej Karpathy's lecture on neural networks ([Link](https://www.youtube.com/watch?v=VMj-3S1tku0))

## What's being implemented?
- A tiny scalar auto gradient engine
- Small neural network trained using it

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import random
%matplotlib inline

## Functions and derivatives


In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)

In [ ]:
h = 0.00000001
x = -2.0
(f(x + h) - f(x))/h

### Derivatives across multiple variables


In [ ]:
a = 2.0
b = -3.0
c = 10.0

In [ ]:
h = 0.0001

d1 = a*b + c

c += h
d2 = a*b + c

print("d1", d1)
print("d2", d2)
print("slope", (d2 - d1)/h)

## Components of micrograd


In [ ]:
class Value:

    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda : None

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        
        out = Value(self.data * other.data, (self,other))

        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += out.grad * other * (self.data**(other - 1))
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * other**-1

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1)/ (math.exp(2 * x) + 1)
        out = Value(t, (self,))

        def _backward():
            self.grad += out.grad * (1 - t**2)
        out._backward = _backward

        return out

    def zero_grad(self):
        self.grad = 0.0

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
        

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

In [ ]:
d = a*b + c

In [ ]:
# Skipping part of graph viz and deriving the logic/intuition for back propagation algorithm
# Skipping part on tanh as activation and computing the gradients of an expression using tanh
# Skipping section on pytorch

In [ ]:
d.backward()

In [ ]:
print('grad(d)', d.grad)
print('grad(c)', c.grad)
print('grad(b)', b.grad)
print('grad(a)', a.grad)

In [ ]:
a = Value(2.0)
b = Value(5.0)
e = a / b  
e.backward()

print('e', e.data)
print('grad(e)', e.grad)
print('grad(b)', b.grad)
print('grad(a)', a.grad)

## Building Neural Nets


In [ ]:
class Neuron:

    def __init__(self, nin, learning_rate):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
        self._learning_rate = learning_rate

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()

        return out

    def _parameters(self):
        return self.w +  [self.b]

    def zero_grad(self):
        params = self._parameters()
        for p in params:
            p.zero_grad()

    def update(self):
        params = self._parameters()

        for p in params:
            p.data += -1 * self._learning_rate * p.grad


class Layer:
    def __init__(self, nin, nout, learning_rate):
        self.neurons = [Neuron(nin, learning_rate) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def zero_grad(self):
        for neuron in self.neurons:
            neuron.zero_grad()

    def update(self):
        for neuron in self.neurons:
            neuron.update()

class MLP:
    def __init__(self, nin, nouts, learning_rate):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], learning_rate) for i in range(len(nouts))]

    def __call__(self, x):
        out = x
        for layer in self.layers:
            out = layer(out)

        return out

    def zero_grad(self):
        for layer in self.layers:
            layer.zero_grad()

    def update(self):
        for layer in self.layers:
            layer.update()

In [ ]:
x = [2.0, 3.0]
n = Neuron(2, 0.01)
n(x)

In [ ]:
mlp = MLP(3, [4, 5, 1], 0.01)

In [ ]:
x = [2.0, -1.0, 3.14]
mlp(x)

# Testing our neural net

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

In [ ]:
ypred = [mlp(x) for x in xs]
ypred

## Loss - capturing performance of neural net predictions

In [ ]:
loss = sum((- ygt + yout)**2 for ygt, yout in zip(ys, ypred))
loss

In [ ]:
mlp.zero_grad()

mlp.layers[0].neurons[0].w[0].grad

In [ ]:
loss.backward()

mlp.layers[0].neurons[0].w[0].grad

## Running gradient descent

In [ ]:
mlp.layers[0].neurons[0].w[0].data, mlp.layers[0].neurons[0].w[0].grad

In [ ]:
mlp.update()

In [ ]:
mlp.layers[0].neurons[0].w[0].data, mlp.layers[0].neurons[0].w[0].grad

### putting it altogether
    

In [ ]:
mlp = MLP(3, [4, 5, 1], 0.05)

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [-1.0, 1.0, 1.0, -1.0]

In [ ]:
for k in range(50):
    ypred = [mlp(x) for x in xs]

    loss = sum((- ygt + yout)**2 for ygt, yout in zip(ys, ypred))

    mlp.zero_grad()
    loss.backward()

    mlp.update()

    print(k, loss)

In [ ]:
ypred

# Py Torch demo

In [ ]:
import torch

In [ ]:
x1 = torch.Tensor([2.0]).double(); x1.requires_grad = True
x2 = torch.Tensor([0.0]).double(); x2.requires_grad = True

w1 = torch.Tensor([-3.0]).double(); w1.requires_grad = True
w2 = torch.Tensor([1.0]).double(); w2.requires_grad = True

b = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True

n = x1 * w1 + x2 * w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('----')
print('x2', x2.grad.item())
print('x1', x1.grad.item())
print('w2', w2.grad.item())
print('w1', w1.grad.item())
print('b', b.grad.item())